# 03b – Support Vector Regression (SVR)
**Owner:** Member 3 (Classical Machine Learning Engineer)

Builds an `SVR` inside a scikit-learn `Pipeline`, tunes hyperparameters (`C`, `epsilon`, `kernel`, `gamma`) using `GridSearchCV` on training data only, and evaluates on validation and test splits.

In [1]:
import sys
import time
from pathlib import Path

import joblib
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CV_FOLDS, MODELS_DIR, RANDOM_SEED
from src.evaluate import (
    compute_metrics,
    cross_validate_model,
    plot_parity,
    plot_residuals,
    save_model_run_metadata,
    save_predictions,
)
from src.features import build_preprocessor, get_xy

MODELS_DIR.mkdir(parents=True, exist_ok=True)
X_train, y_train, id_train = get_xy("train")
X_val, y_val, id_val = get_xy("val")
X_test, y_test, id_test = get_xy("test")

In [2]:
svr_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", SVR()),
])

param_grid = [
    {
        "model__kernel": ["rbf"],
        "model__C": [1.0, 5.0, 10.0, 25.0],
        "model__epsilon": [0.05, 0.1, 0.2],
        "model__gamma": ["scale", 0.01, 0.005],
    },
    {
        "model__kernel": ["linear"],
        "model__C": [0.5, 1.0, 5.0],
        "model__epsilon": [0.1],
    },
]

cv_split = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
search = GridSearchCV(
    estimator=svr_pipe,
    param_grid=param_grid,
    cv=cv_split,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    refit=False,
    verbose=1,
)

t_search_start = time.perf_counter()
search.fit(X_train, y_train)
search_time = time.perf_counter() - t_search_start

print(f"Search completed in {search_time:.2f} s")
print("Best CV RMSE:", round(-search.best_score_, 4))
print("Best Params :", search.best_params_)

Fitting 5 folds for each of 39 candidates, totalling 195 fits


Search completed in 173.54 s
Best CV RMSE: 1.1005
Best Params : {'model__C': 10.0, 'model__epsilon': 0.2, 'model__gamma': 0.01, 'model__kernel': 'rbf'}


In [3]:
# Build finalized pipeline with best hyperparameters and measure fit time
best_svr_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
best_svr_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", SVR(**best_svr_params)),
])

cv_summary = cross_validate_model(best_svr_pipe, X_train, y_train, cv=CV_FOLDS)
print("Final SVR 5-Fold CV:", {k: round(v, 4) for k, v in cv_summary.items()})

t_fit_start = time.perf_counter()
best_svr_pipe.fit(X_train, y_train)
fit_time = time.perf_counter() - t_fit_start

val_pred = best_svr_pipe.predict(X_val)
val_metrics = compute_metrics(y_val, val_pred)
print(f"Validation Metrics: {val_metrics}")

# Finalized model evaluation on Test set
test_pred = best_svr_pipe.predict(X_test)
test_metrics = compute_metrics(y_test, test_pred)
print(f"Test Metrics      : {test_metrics}")

joblib.dump(best_svr_pipe, MODELS_DIR / "svr_best.joblib")
save_predictions(id_val, y_val, val_pred, "svr", "val")
save_predictions(id_test, y_test, test_pred, "svr", "test")
save_model_run_metadata(
    model_name="svr",
    display_name="SVR",
    training_time_sec=fit_time,
    search_time_sec=search_time,
    cv_summary=cv_summary,
    extra={"best_params": search.best_params_},
)

plot_parity(y_test, test_pred, "svr", split="test")
plot_residuals(y_test, test_pred, "svr", split="test")
print(f"SVR training time: {fit_time:.2f} s | search time: {search_time:.2f} s")

Final SVR 5-Fold CV: {'r2_mean': 0.4826, 'r2_std': 0.0285, 'rmse_mean': 1.1005, 'rmse_std': 0.0332, 'mae_mean': 0.7578, 'mae_std': 0.0164}


Validation Metrics: {'r2': 0.5540624448416003, 'rmse': 1.1209030963649536, 'mae': 0.75728602929285}
Test Metrics      : {'r2': 0.5010790927891413, 'rmse': 1.033452555268656, 'mae': 0.7099520737166268}


SVR training time: 1.13 s | search time: 173.54 s
